In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py 
import sys 
import os, glob
from datetime import datetime
from zoneinfo import ZoneInfo

# Path to repo root (two directories above notebook)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(repo_root)


In [2]:
file='/global/cfs/cdirs/dune/users/lmlepin/neutron_source_CL_data/source_ambe_bin3/two_trig_9600ns_window/mpd_run_run2data_rctl_618_p284_v0p2.FLOW.hdf5'
h5_file = h5py.File(file,'r') 
print(f"This file keys: {h5_file.keys()}")

This file keys: <KeysViewHDF5 ['charge', 'combined', 'geometry_info', 'lar_info', 'light', 'run_info']>


In [9]:
print(f"Entries: {h5_file['light/events/data'].dtype.names}")
t_start = h5_file['light/events/data']['tai_ns'][3][0]
t_end = h5_file['light/events/data']['tai_ns'][4][0]
print(t_start)
print(t_end)
print(f"Delta T: {round((t_end - t_start)*1e-3)}")

Entries: ('id', 'event', 'sn', 'utime_ms', 'tai_ns', 'wvfm_valid', 'trig_type')
1762624135913934336
1762624135914014208
Delta T: 80


In [ ]:
def get_time_stamps(h5_file,debug=True):
    t_start = h5_file['light/events/data']['utime_ms'][0][0]
    t_end = h5_file['light/events/data']['utime_ms'][-1][0]
    if(debug):
        print(f"First trigger unix time: {t_start}")
        print(f"First trigger local time: {datetime.fromtimestamp(t_start/1e3, tz=ZoneInfo('America/Chicago'))}")
        print(f"Last trigger unix time: {t_end}")
        print(f"Last trigger local time: {datetime.fromtimestamp(t_end/1e3, tz=ZoneInfo('America/Chicago'))}")
    return t_start 



In [5]:
output_txt = "AmBe_mod2_pmt_trigger_100us_period.txt"

file_index = 0
chicago = ZoneInfo("America/Chicago")
utc = ZoneInfo("UTC")

start_local = datetime(2025, 11, 3, 16, 40, 0, tzinfo=chicago)
end_local   = datetime(2025, 11, 3, 21, 30, 0, tzinfo=chicago)

start_ts = start_local.astimezone(utc).timestamp()
end_ts   = end_local.astimezone(utc).timestamp()

print(start_ts)
print(end_ts)

pattern = "/global/cfs/cdirs/dune/users/lmlepin/neutron_source_CL_data/source_ambe_bin0/mod2_pmt_trig/*"

with open(output_txt, "w") as f_out:
    for ifile in glob.iglob(pattern):
        if file_index % 40 == 0:
            print(ifile)

        try:
            with h5py.File(ifile, "r") as this_file:
                this_t = get_time_stamps(this_file, False)

            # if this_t is in milliseconds
            this_t_sec = this_t / 1000.0

            if start_ts <= this_t_sec <= end_ts:
                f_out.write(ifile + "\n")

        except Exception as e:
            print(f"Error processing {ifile}: {e}")

        file_index += 1



1762209600.0
1762227000.0
/global/cfs/cdirs/dune/users/lmlepin/neutron_source_CL_data/source_ambe_bin0/mod2_pmt_trig/mpd_run_run2data_rctl_557_p23_v0p2.FLOW.hdf5


KeyboardInterrupt: 